# Phase 1 — Data Cleaning & Feature Engineering

**Source:** Strong app CSV export  
**Goal:** Transform raw, set-level workout logs into two clean, analysis-ready datasets.

**Output:**
- `data/clean/clean_sets.csv` — one row per working set, with engineered features
- `data/clean/workout_summary.csv` — one row per workout, aggregated metrics

---
### Notebook structure
1. Setup & Load
2. Exploratory Inspection
3. Filter & Clean
4. Feature Engineering
5. Workout-Level Aggregation
6. Export

## 1. Setup & Load

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

In [2]:
# Resolve raw file — look in data/raw first, fall back to project root
_candidates = [
    Path("../data/raw/strong_userdata.csv"),
    Path("../strong_userdata.csv"),
]
RAW_FILE = next((p for p in _candidates if p.exists()), None)
assert RAW_FILE is not None, "strong_userdata.csv not found. Place it in data/raw/ or the project root."

CLEAN_DIR = Path("../data/clean")
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

print(f"Raw file : {RAW_FILE.resolve()}")
print(f"Clean dir: {CLEAN_DIR.resolve()}")

Raw file : C:\projects\FitnessData\data\raw\strong_userdata.csv
Clean dir: C:\projects\FitnessData\data\clean


In [3]:
# Strong exports use semicolons as delimiters and wraps all values in double quotes
raw = pd.read_csv(
    RAW_FILE,
    sep=";",
    quotechar='"',
    encoding="utf-8",
    dtype=str,          # load everything as string first — we cast deliberately below
    na_values=[""],
)

print(f"Rows: {len(raw):,}  |  Columns: {raw.shape[1]}")
raw.head(3)

Rows: 23,058  |  Columns: 13


,Workout #,Date,Workout Name,Duration (sec),Exercise Name,Set Order,Weight (kg),Reps,RPE,Distance (meters),Seconds,Notes,Workout Notes
0,1,2020-05-28 12:58:47,Day 3,4195,Incline Bench Press (Barbell),W,40.0,15,NaN,NaN,NaN,NaN,NaN
1,1,2020-05-28 12:58:47,Day 3,4195,Incline Bench Press (Barbell),1,60.0,7,NaN,NaN,NaN,NaN,NaN
2,1,2020-05-28 12:58:47,Day 3,4195,Incline Bench Press (Barbell),2,60.0,8,NaN,NaN,NaN,NaN,NaN


## 2. Exploratory Inspection

Before any transformation, understand the shape and quality of the data.

In [4]:
raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23058 entries, 0 to 23057
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Workout #          23058 non-null  object
 1   Date               23058 non-null  object
 2   Workout Name       23058 non-null  object
 3   Duration (sec)     23058 non-null  object
 4   Exercise Name      23058 non-null  object
 5   Set Order          23058 non-null  object
 6   Weight (kg)        19203 non-null  object
 7   Reps               19287 non-null  object
 8   RPE                0 non-null      object
 9   Distance (meters)  23 non-null     object
 10  Seconds            3772 non-null   object
 11  Notes              9 non-null      object
 12  Workout Notes      378 non-null    object
dtypes: object(13)
memory usage: 2.3+ MB


In [5]:
# Null count per column — helps decide which columns are usable
null_pct = (raw.isnull().sum() / len(raw) * 100).round(1)
null_pct.rename("null_%").to_frame()

,null_%
Workout #,0.00
Date,0.00
Workout Name,0.00
Duration (sec),0.00
Exercise Name,0.00
Set Order,0.00
Weight (kg),16.70
Reps,16.40
RPE,100.00
Distance (meters),99.90


In [6]:
# 'Set Order' drives the row type — inspect every distinct value
# Numeric values = working sets; 'W' = warmup; others = metadata rows
print(raw["Set Order"].value_counts(dropna=False).to_string())

Set Order
1             5384
2             5295
3             5264
Rest Timer    3661
4             2202
5              652
W              237
6              193
D              133
7               24
Note             9
8                1
9                1
10               1
11               1


In [7]:
# Date range of the dataset
dates = pd.to_datetime(raw["Date"], errors="coerce")
print(f"Earliest workout : {dates.min().date()}")
print(f"Latest workout   : {dates.max().date()}")
print(f"Span             : {(dates.max() - dates.min()).days} days")

Earliest workout : 2020-05-28
Latest workout   : 2026-05-27
Span             : 2190 days


In [8]:
# Top 20 exercises by raw row count (includes warmups/notes, for orientation)
raw["Exercise Name"].value_counts().head(20)

Exercise Name
Incline Bench Press (Dumbbell)             1014
Lateral Raise (Dumbbell)                    965
Bench Press (Barbell)                       868
Cable Crossover                             746
Lat Pulldown (Cable)                        733
Leg Extension (Machine)                     726
Pull Up                                     723
Romanian Deadlift (Barbell)                 709
Chest Dip                                   653
Bicep Curl (Cable)                          642
Hack Squat                                  556
Seated Leg Curl (Machine)                   548
Pullover (Machine)                          539
Seated Overhead Press (Dumbbell)            528
Seated Leg Press (Machine)                  499
Triceps Extension (Cable)                   494
Lying Leg Curl (Machine)                    477
Shrug (Dumbbell)                            470
Triceps Pushdown (Cable - Straight Bar)     456
Reverse Fly (Machine)                       455
Name: count, dtype: int64

## 3. Filter & Clean

The raw file contains three types of non-set rows that must be removed before analysis:

| `Set Order` value | Meaning |
|---|---|
| `W` | Warmup set (tracked separately by Strong) |
| `Rest Timer` | Auto-logged rest period — no weight/reps data |
| `Note` | Free-text note attached to an exercise |

We keep only rows where `Set Order` is a positive integer.

In [9]:
# Standardise column names to snake_case before any further work
COLUMN_MAP = {
    "Workout #":        "workout_id",
    "Date":             "date",
    "Workout Name":     "workout_name",
    "Duration (sec)":   "duration_sec",
    "Exercise Name":    "exercise_name",
    "Set Order":        "set_order",
    "Weight (kg)":      "weight_kg",
    "Reps":             "reps",
    "RPE":              "rpe",
    "Distance (meters)": "distance_m",
    "Seconds":          "seconds",
    "Notes":            "notes",
    "Workout Notes":    "workout_notes",
}

df = raw.rename(columns=COLUMN_MAP)

In [10]:
# Keep only rows where set_order is a pure integer string (e.g. '1', '2', '3')
# This is more robust than an exclusion list — it survives any new metadata row types Strong might add
is_working_set = df["set_order"].str.match(r"^\d+$", na=False)

df_sets = df.loc[is_working_set].copy()

dropped = len(df) - len(df_sets)
print(f"Rows removed (warmups, rest timers, notes): {dropped:,}")
print(f"Working sets remaining: {len(df_sets):,}")

Rows removed (warmups, rest timers, notes): 4,040
Working sets remaining: 19,018


In [11]:
# Cast each column to its correct type
# errors='coerce' turns unparseable values into NaN rather than raising — we inspect those after
df_sets = df_sets.assign(
    date         = pd.to_datetime(df_sets["date"], errors="coerce"),
    workout_id   = pd.to_numeric(df_sets["workout_id"], errors="coerce").astype("Int64"),
    duration_sec = pd.to_numeric(df_sets["duration_sec"], errors="coerce").astype("Int64"),
    set_order    = pd.to_numeric(df_sets["set_order"], errors="coerce").astype("Int64"),
    weight_kg    = pd.to_numeric(df_sets["weight_kg"], errors="coerce"),
    reps         = pd.to_numeric(df_sets["reps"], errors="coerce").astype("Int64"),
    rpe          = pd.to_numeric(df_sets["rpe"], errors="coerce"),
)

In [12]:
# Inspect any rows where weight or reps could not be parsed — these would skew calculations
missing_core = df_sets[df_sets["weight_kg"].isna() | df_sets["reps"].isna()]
print(f"Rows with missing weight or reps: {len(missing_core):,}")
missing_core.head(10)

Rows with missing weight or reps: 185


,workout_id,date,workout_name,duration_sec,exercise_name,set_order,weight_kg,reps,rpe,distance_m,seconds,notes,workout_notes
2646,126,2021-01-12 13:42:41,Push Home,5373,Bike ride,1,NaN,<NA>,NaN,0.0,1320.0,NaN,NaN
2653,127,2021-01-13 13:48:42,Home Core,4957,Plank,1,NaN,<NA>,NaN,NaN,60.0,NaN,NaN
2654,127,2021-01-13 13:48:42,Home Core,4957,Plank,2,NaN,<NA>,NaN,NaN,60.0,NaN,NaN
2655,127,2021-01-13 13:48:42,Home Core,4957,Plank,3,NaN,<NA>,NaN,NaN,60.0,NaN,NaN
2656,127,2021-01-13 13:48:42,Home Core,4957,Side Plank,1,NaN,<NA>,NaN,NaN,45.0,NaN,NaN
2657,127,2021-01-13 13:48:42,Home Core,4957,Side Plank,2,NaN,<NA>,NaN,NaN,45.0,NaN,NaN
2658,127,2021-01-13 13:48:42,Home Core,4957,Side Plank,3,NaN,<NA>,NaN,NaN,45.0,NaN,NaN
2659,127,2021-01-13 13:48:42,Home Core,4957,L Sit,1,NaN,<NA>,NaN,NaN,20.0,NaN,NaN
2660,127,2021-01-13 13:48:42,Home Core,4957,L Sit,2,NaN,<NA>,NaN,NaN,20.0,NaN,NaN
2661,127,2021-01-13 13:48:42,Home Core,4957,L Sit,3,NaN,<NA>,NaN,NaN,20.0,NaN,NaN


In [13]:
# Drop rows without usable weight or reps — they cannot contribute to any metric
df_sets = df_sets.dropna(subset=["weight_kg", "reps"]).copy()

# Sanity check: no negative weights or reps
assert (df_sets["weight_kg"] >= 0).all(), "Negative weight values found"
assert (df_sets["reps"] > 0).all(), "Zero or negative rep values found"

print(f"Clean working sets: {len(df_sets):,}")

Clean working sets: 18,833


### Outlier Removal — Reps

Data entry errors in Strong produce physiologically impossible rep counts (e.g. `16814`).
In weighted strength training, sets above **100 reps** are effectively impossible and indicate a typo.
We log every affected row before dropping it so the removal is transparent and reproducible.

In [14]:
# Inspect the upper tail of the reps distribution
print(df_sets["reps"].describe(percentiles=[0.25, 0.5, 0.75, 0.95, 0.99]).to_string())
print()
print("Top 10 highest-rep sets:")
print(df_sets.nlargest(10, "reps")[["date", "exercise_name", "weight_kg", "reps"]].to_string(index=False))

count   18833.00
mean       14.08
std       122.49
min         1.00
25%        11.00
50%        13.00
75%        16.00
95%        20.00
99%        22.00
max     16814.00

Top 10 highest-rep sets:
               date           exercise_name  weight_kg  reps
2021-01-21 12:15:05   Bicep Curl (Dumbbell)       5.00 16814
2021-05-17 17:01:53  Hip Abductor (Machine)      57.50   146
2021-02-21 12:21:44      Bicep Curl (Cable)       0.00   114
2026-05-25 10:18:30        Wrist Curl Cable      25.00    33
2024-04-07 19:42:44 Leg Extension (Machine)      45.00    31
2020-12-03 13:10:40        Kettlebell Swing      16.00    30
2021-03-12 13:29:55       Cross Body Crunch       0.00    30
2021-03-12 13:29:55                  Crunch       0.00    30
2021-03-15 15:40:01       Cross Body Crunch       0.00    30
2021-03-15 15:40:01       Cross Body Crunch       0.00    30


In [15]:
MAX_REPS = 100  # physiological ceiling for weighted strength training sets

outliers = df_sets[df_sets["reps"] > MAX_REPS]
if len(outliers) > 0:
    print(f"Removing {len(outliers)} row(s) with reps > {MAX_REPS}:")
    print(outliers[["date", "exercise_name", "weight_kg", "reps"]].to_string(index=False))
else:
    print("No rep outliers found.")

df_sets = df_sets[df_sets["reps"] <= MAX_REPS].copy()
print(f"\nWorking sets after outlier removal: {len(df_sets):,}")

Removing 3 row(s) with reps > 100:
               date          exercise_name  weight_kg  reps
2021-01-21 12:15:05  Bicep Curl (Dumbbell)       5.00 16814
2021-02-21 12:21:44     Bicep Curl (Cable)       0.00   114
2021-05-17 17:01:53 Hip Abductor (Machine)      57.50   146

Working sets after outlier removal: 18,830


## 4. Feature Engineering

### Volume
**Volume** (`weight × reps`) is the standard measure of total mechanical work per set.
Summed across a workout it gives *total volume load* — a key proxy for training stimulus.

### Estimated 1RM (Epley Formula)
The **one-rep maximum** is the gold standard for comparing strength over time.
Because you rarely test a true 1RM, we estimate it from sub-maximal sets:

$$\hat{1RM} = \text{weight} \times \left(1 + \frac{\text{reps}}{30}\right)$$

For single-rep sets the formula slightly overestimates, so we return the raw weight instead.

In [16]:
df_sets = df_sets.assign(
    # Total mechanical work for this set
    volume_kg = df_sets["weight_kg"] * df_sets["reps"],

    # Epley 1RM estimate — clamp to actual weight for single-rep sets
    estimated_1rm = np.where(
        df_sets["reps"] == 1,
        df_sets["weight_kg"],
        df_sets["weight_kg"] * (1 + df_sets["reps"] / 30),
    ),

    # Calendar columns — useful for grouping and trend analysis later
    year  = df_sets["date"].dt.year,
    month = df_sets["date"].dt.to_period("M").astype(str),
    week  = df_sets["date"].dt.to_period("W").astype(str),
    day_of_week = df_sets["date"].dt.day_name(),
)

In [17]:
# Spot-check: top estimated 1RM, one entry per exercise
# Verifies the Epley formula produces plausible results across different movements
sample = (
    df_sets
    .sort_values("estimated_1rm", ascending=False)
    .drop_duplicates(subset="exercise_name")
    .head(5)
    [["date", "exercise_name", "weight_kg", "reps", "estimated_1rm"]]
)
sample

,date,exercise_name,weight_kg,reps,estimated_1rm
886,2020-08-05 14:32:19,Leg Press,250.00,16,383.33
14856,2024-08-16 14:38:24,Hip Adductor (Machine),117.50,23,207.58
587,2020-07-15 15:27:35,Seated Leg Press (Machine),137.50,10,183.33
7483,2022-04-11 09:59:29,Leg Extension (Machine),95.00,22,164.67
14056,2024-06-03 15:30:05,Triceps Extension (Machine),102.50,18,164.00


## 5. Workout-Level Aggregation

Aggregate the set-level data into one row per workout.
This `workout_summary` table is the primary input for time-series and trend analysis.

In [18]:
workout_summary = (
    df_sets
    .groupby(["workout_id", "date", "workout_name", "duration_sec"], dropna=False)
    .agg(
        total_sets        = ("set_order",     "count"),
        total_volume_kg   = ("volume_kg",     "sum"),
        peak_estimated_1rm= ("estimated_1rm", "max"),
        unique_exercises  = ("exercise_name", "nunique"),
    )
    .reset_index()
    .assign(
        duration_min = lambda x: (x["duration_sec"] / 60).round(1),
    )
    .drop(columns="duration_sec")
    .sort_values("date")
    .reset_index(drop=True)
)

print(f"Workouts: {len(workout_summary):,}")
workout_summary.head(5)

Workouts: 849


,workout_id,date,workout_name,total_sets,total_volume_kg,peak_estimated_1rm,unique_exercises,duration_min
0,1,2020-05-28 12:58:47,Day 3,19,8557.50,260.00,6,69.90
1,2,2020-05-30 16:14:11,Day 4,6,3090.00,71.75,2,74.20
2,3,2020-06-01 10:48:59,Day 5,7,2396.25,88.67,2,67.30
3,4,2020-06-03 16:02:10,Day 1,22,13864.00,273.33,7,79.60
4,5,2020-06-05 17:21:14,Day 2,19,14442.00,316.67,6,81.60


In [19]:
# Quick descriptive summary — validates that aggregated numbers are in a plausible range
workout_summary[["duration_min", "total_sets", "total_volume_kg", "unique_exercises"]].describe().round(1)

,duration_min,total_sets,total_volume_kg,unique_exercises
count,849.00,849.00,849.00,849.00
mean,75.70,22.20,11046.00,6.20
std,17.20,5.10,6699.00,1.30
min,0.60,3.00,0.00,1.00
25%,66.60,19.00,6175.50,6.00
50%,75.90,23.00,10039.20,6.00
75%,84.40,26.00,15566.50,7.00
max,182.60,36.00,35145.00,9.00


## 6. Export

Write both datasets to `data/clean/`. These files are the single source of truth for all downstream analysis (SQL, Power BI, statistics notebook).

In [20]:
# Select and order columns for the sets export
SETS_COLUMNS = [
    "workout_id", "date", "workout_name", "year", "month", "week", "day_of_week",
    "exercise_name", "set_order", "weight_kg", "reps", "rpe",
    "volume_kg", "estimated_1rm",
    "notes",
]

clean_sets = df_sets[SETS_COLUMNS].sort_values(["date", "workout_id", "exercise_name", "set_order"])

clean_sets_path = CLEAN_DIR / "clean_sets.csv"
clean_sets.to_csv(clean_sets_path, index=False)
print(f"Exported {len(clean_sets):,} rows → {clean_sets_path}")

Exported 18,830 rows → ..\data\clean\clean_sets.csv


In [21]:
workout_summary_path = CLEAN_DIR / "workout_summary.csv"
workout_summary.to_csv(workout_summary_path, index=False)
print(f"Exported {len(workout_summary):,} rows → {workout_summary_path}")

Exported 849 rows → ..\data\clean\workout_summary.csv


In [22]:
# Final summary — confirm output looks correct before handing off to Phase 2
print("=== clean_sets.csv ===")
print(f"  Rows         : {len(clean_sets):,}")
print(f"  Date range   : {clean_sets['date'].min().date()} → {clean_sets['date'].max().date()}")
print(f"  Exercises    : {clean_sets['exercise_name'].nunique()}")
print()
print("=== workout_summary.csv ===")
print(f"  Rows         : {len(workout_summary):,}")
print(f"  Avg duration : {workout_summary['duration_min'].mean():.1f} min")
print(f"  Avg volume   : {workout_summary['total_volume_kg'].mean():,.0f} kg")
print(f"  Avg sets     : {workout_summary['total_sets'].mean():.1f}")

=== clean_sets.csv ===
  Rows         : 18,830
  Date range   : 2020-05-28 → 2026-05-27
  Exercises    : 120

=== workout_summary.csv ===
  Rows         : 849
  Avg duration : 75.7 min
  Avg volume   : 11,046 kg
  Avg sets     : 22.2
